#**Telecom X - Análise de Evasão de Clientes. (Parte 2)**

##Importando as bibliotecas

###📌 Extração (coleta)

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import chi2
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

CarregueI o arquivo CSV que contém os dados tratados anteriormente.

In [2]:
df = pd.read_csv('/content/dados_tratados (1).csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/dados_tratados (1).csv'

In [ ]:
df.info()

✔️ Verificação novamente de valore nulos

Verifiquei novamente a presença de dados nulos

In [ ]:
df.isnull().sum()

In [ ]:
df.columns

##🛠️ Pre processamento

✔️ Remover colunas irrelevantes

Eliminei colunas que não trazem valor para a análise ou para os modelos preditivos, como identificadores únicos (por exemplo, o ID do cliente). Essas colunas não ajudam na previsão da evasão e podem até prejudicar o desempenho dos modelos.

In [ ]:
df = df.drop(columns=['CustomerID',"ChargesDaily", "ChargesTotal"])
df

✔️ Agrupamentode **No** e **No service**

In [ ]:
cols_to_fix = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

for col in cols_to_fix:
    df[col] = df[col].replace( 'No internet service', 'No')

## 📊 EDA

✔️ Correlação entre variaveis numericas

In [ ]:
# Selecionar apenas variáveis numéricas
df_numeric = df.select_dtypes(include=["int64", "float64"])

# Converter Churn para numérico
df_numeric["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})


# Calcular matriz de correlação
corr_matrix = df_numeric.corr()

# Visualizar heatmap
plt.figure(figsize=(10,6))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlação - Variáveis Numéricas")
plt.show()

2️⃣ Análise de churn por categoria (variáveis categóricas)

🔹 Selecionar variáveis categóricas

In [ ]:
categorical_cols = df.select_dtypes(include=["object"]).columns
categorical_cols = categorical_cols.drop("Churn")

🔹 Tabela de proporção de churn por categoria

In [ ]:
for col in categorical_cols:
    print(f"\n📌 Análise da variável: {col}")
    #Ele cria uma tabela de porcentagem de churn dentro de cada categoria de uma variável.
    display(pd.crosstab(df[col], df["Churn"], normalize="index") * 100)


In [ ]:
for col in categorical_cols:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=col, hue="Churn")
    plt.title(f"Distribuição de Churn por {col}")
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
# Criar cópia
df_rank = df.copy()

# Converter Churn para binário
df_rank["Churn"] = df_rank["Churn"].map({"Yes": 1, "No": 0})

# Separar variáveis categóricas
categorical_cols = df_rank.select_dtypes(include=["object"]).columns

# Aplicar Label Encoding temporário (apenas para teste estatístico)
le = LabelEncoder()
for col in categorical_cols:
    df_rank[col] = le.fit_transform(df_rank[col])

# Separar X e y
X_cat = df_rank[categorical_cols]
y = df_rank["Churn"]

# Aplicar teste Qui-quadrado
chi_scores, p_values = chi2(X_cat, y)

# Criar DataFrame com ranking
chi_df = pd.DataFrame({
    "Variável": categorical_cols,
    "Chi2 Score": chi_scores,
    "p-value": p_values
})

# Ordenar por importância
chi_df = chi_df.sort_values(by="Chi2 Score", ascending=False)

chi_df


In [ ]:
significant_vars = chi_df[chi_df["p-value"] < 0.05]
print("\n✅ Variables significativas (p < 0.05):")
display(significant_vars)

In [ ]:
df = df.drop(columns=["Gender", "PhoneService"])

## 🎯 Split

1. Separação de X e Y

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]


2. Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


3.One-Hot Encoding

In [ ]:
# Aplicar One-Hot Encoding no treino
X_train = pd.get_dummies(X_train, drop_first=True)

# Aplicar no teste
X_test = pd.get_dummies(X_test, drop_first=True)

# Garantir que treino e teste tenham as mesmas colunas
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [ ]:
X_train.head()

4. Verificar desbalaceamento

In [ ]:
y_train.value_counts(normalize=True) * 100

##Treinamento


Modelo 1 — Regressão Logística

In [ ]:


log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log_normal = log_model.predict(X_test)

print("Regressão Logística")
print(classification_report(y_test, y_pred_log_normal))

Modelo 2 — Random Forest

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Random Forest")
print(classification_report(y_test, y_pred_rf))

In [ ]:
log_model = LogisticRegression(max_iter=1000, class_weight="balanced")
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print(classification_report(y_test, y_pred_log))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_log)

# Plot
plt.figure()
plt.imshow(cm)
plt.title("Matriz de Confusão - Regressão Logística (Balanceada)")
plt.xlabel("Predito")
plt.ylabel("Real")

# Colocar números dentro da matriz
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xticks([0,1], ["No", "Yes"])
plt.yticks([0,1], ["No", "Yes"])

plt.show()

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_log_normal)

# Plot
plt.figure()
plt.imshow(cm)
plt.title("Matriz de Confusão - Regressão Logística (Balanceada)")
plt.xlabel("Predito")
plt.ylabel("Real")

# Colocar números dentro da matriz
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.xticks([0,1], ["No", "Yes"])
plt.yticks([0,1], ["No", "Yes"])

plt.show()

#Relatório Final

Nesta etapa elaborei um relatório detalhado, destacando os fatores que mais influenciam a evasão, com base nas variáveis selecionadas e no desempenho de cada modelo.

Identifiquei os principais fatores que afetam a evasão de clientes e propus estratégias de retenção com base nos resultados obtidos.



     
##Desempenho dos Modelos

Entre os modelos avaliados, a Regressão Logística apresentou o melhor desempenho geral, considerando:

Acurácia de aproximadamente 80%
Melhor equilíbrio entre precisão e recall
Melhor F1-score para a classe de cancelamento
Maior interpretabilidade dos resultados
O Random Forest também apresentou bom desempenho, porém com recall inferior para a classe de cancelamento, o que é um ponto crítico quando o objetivo é prever evasão.

##Principais Fatores que Influenciam a Evasão.

A análise dos coeficientes da Regressão Logística e da importância das variáveis no Random Forest mostrou que os principais fatores associados ao cancelamento são:

##Fatores que aumentam a evasão:

Contrato mensal (Month-to-month)
Internet do tipo Fiber optic
Pagamento via Electronic check
Fatura digital
Ausência de serviços como segurança online
Alto valor total acumulado
Baixo tempo de contrato
Fatores que reduzem a evasão:

Maior tempo de permanência na empresa (principal fator)
Contratos de dois anos
Presença de suporte técnico
Métodos automáticos de pagamento (cartão ou transferência)
O tempo de contrato foi a variável mais relevante no modelo, indicando que clientes novos possuem maior risco de cancelamento.

##Interpretação Estratégica.

Os resultados indicam que a evasão está fortemente relacionada à ausência de fidelização contratual e ao baixo tempo de relacionamento com a empresa.

Clientes com contrato mensal possuem maior flexibilidade para cancelar, o que aumenta o risco de churn. Além disso, clientes nos primeiros meses de serviço ainda não consolidaram vínculo com a empresa, tornando-se mais sensíveis a insatisfações.

##Estratégias de Retenção Propostas.

Com base nos resultados obtidos, recomenda-se:

01 - Incentivar contratos de longo prazo: Oferecer descontos ou benefícios para migração de contrato mensal para contratos anuais ou bianuais.

02 - Foco nos primeiros meses do cliente: Criar campanhas de acompanhamento nos primeiros 3–6 meses, período de maior risco.

03 - Incentivar métodos automáticos de pagamento: Estimular pagamento via cartão ou débito automático para reduzir cancelamentos associados ao Electronic check.

04 - Oferecer suporte técnico como benefício estratégico: Clientes com suporte técnico apresentam menor probabilidade de evasão.

05 - Monitoramento contínuo com modelo preditivo: Utilizar a Regressão Logística como ferramenta operacional para identificar clientes com alto risco de churn e agir preventivamente.

...

##Conclusão Final.

O projeto demonstrou que é possível prever a evasão de clientes com bom nível de precisão utilizando técnicas de Machine Learning.

A Regressão Logística mostrou-se o modelo mais adequado por combinar desempenho satisfatório e alta interpretabilidade, permitindo não apenas prever cancelamentos, mas também entender os fatores que influenciam essa decisão.

Os resultados obtidos fornecem direcionamentos claros para estratégias de retenção, permitindo que a empresa atue de forma preventiva e orientada por dados.